<a href="https://colab.research.google.com/github/tnkkzk0322/keiba-horse-profile-extractor/blob/main/%E6%8E%A8%E5%A5%A8%E9%A6%AC%E5%88%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libsnappy-dev
!pip -q install numbers-parser pandas plotly

import re
import json
import math
import os
import sys
import time
from itertools import product
from dataclasses import dataclass
from datetime import datetime, date
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import plotly.express as px
from numbers_parser import Document
from IPython.display import display


# --------------------------------------------
# 設定
# --------------------------------------------
numbers_path = "/content/競馬分析.numbers"
sheet_keyword = "予想"

max_budget_per_race = 3000
step = 100
ROI_TARGET_BUDGET_PER_RACE = 10000

MIN_EACH_BY_TYPE = {
    "単勝": 100,
    "馬連": 100,
    "馬単": 100,
    "３連複": 100,
    "３連単": 100,
}

upper_trim_count = 2

# 格×年別ROIは、年ごとの件数が少ないため除外なしで計算
# ただし、買い方は「格ごと最適」ではなく「レース名ごとの最適買い方」を使用する
# 0にすると通常ROIと同じ
RACE_BEST_GRADE_YEAR_UPPER_TRIM_COUNT = 0

# 注力レース定義
# ROI効率を計算した後、
# トリム後ROIが1.5以上かつ的中率が3割を超えているものを注力レースとする
FOCUS_TRIMMED_ROI_MIN = 1.5
FOCUS_TRIMMED_HIT_RATE_MIN = 0.30

SHOW_PROGRESS = True
PROGRESS_INTERVAL_SEC = 1.0

PARALLEL = True
MAX_WORKERS = None
SHOW_PARALLEL_PROGRESS = True

PRIORITY_RACE_NAMES = [
    "東京優駿",
    "目黒記念",
    "葵S"
]

YEARLY_PROFIT_TARGET_YEARS = None

MONTHLY_CUMULATIVE_ROI_YEARS = None
MONTHLY_CUMULATIVE_ROI_MONTHS = list(range(1, 13))

extracted_json_path = "yosou_extracted.json"
ranking_json_path = "ranking_result.json"
all_races_csv_path = "all_races_roi_ranking.csv"
grade_roi_summary_csv_path = "grade_roi_by_race_best_patterns.csv"
grade_year_roi_csv_path = "grade_year_roi_by_race_best_patterns.csv"
grade_year_roi_detail_csv_path = "grade_year_roi_by_race_best_patterns_detail.csv"
monthly_cumulative_trimmed_roi_csv_path = "monthly_cumulative_trimmed_roi.csv"
monthly_cumulative_trimmed_roi_range_csv_path = "monthly_cumulative_trimmed_roi_range.csv"
scatter_html_path = "race_distance_roi_scatter.html"
scatter_source_csv_path = "race_distance_roi_scatter_source.csv"

BET_TYPES = ["単勝", "馬連", "馬単", "３連複", "３連単"]
PAYOUT_COLUMNS = ["単勝", "馬連", "馬単", "３連複", "３連単"]

REQUIRED_COLUMNS = [
    "レース名",
    "日付",
    "距離",
    "格",
    "年齢",
    "単勝",
    "馬連",
    "馬単",
    "３連複",
    "３連単",
    "結果",
]

OPTIONAL_COLUMNS = ["◎", "競馬場", "1着"]

HEADER_ALIASES = {
    "レース名": ["レース名", "レース", "重賞名", "開催名"],
    "日付": ["日付", "開催日", "年月日"],
    "距離": ["距離", "距離m", "距離（m）", "距離(M)", "コース距離"],
    "格": ["格", "クラス", "レース格", "グレード"],
    "年齢": ["年齢", "条件", "出走条件", "年齢条件"],
    "単勝": ["単勝"],
    "馬連": ["馬連"],
    "馬単": ["馬単"],
    "３連複": ["３連複", "3連複"],
    "３連単": ["３連単", "3連単"],
    "結果": ["結果"],
    "1着": ["1着", "１着"],
    "◎": ["◎", "本命"],
    "競馬場": ["競馬場", "競馬場名", "開催場", "場"],
}

RANK_CANDIDATES = ["◯", "▲", "△", "☆", "⑥", "⑦"]


# --------------------------------------------
# ユーティリティ
# --------------------------------------------
def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    s = s.replace("\n", " ").replace("\r", " ")
    s = re.sub(r"\s+", " ", s)
    return s


def zenkaku_to_hankaku_digits(s: str) -> str:
    return s.translate(str.maketrans("０１２３４５６７８９", "0123456789"))


def normalize_header_name(s: str) -> str:
    s = normalize_text(s)
    return s.replace(" ", "").replace("　", "")


def canonical_column_name(col: str) -> Optional[str]:
    col_n = normalize_header_name(col)
    for canon, aliases in HEADER_ALIASES.items():
        for a in aliases:
            if col_n == normalize_header_name(a):
                return canon
    return None


def combine_duplicate_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.columns.is_unique:
        return df

    out = pd.DataFrame(index=df.index)

    for col in dict.fromkeys(df.columns):
        same_cols = df.loc[:, df.columns == col]

        if same_cols.shape[1] == 1:
            out[col] = same_cols.iloc[:, 0]
        else:
            same_cols = same_cols.apply(lambda s: s.map(normalize_text))
            same_cols = same_cols.replace("", pd.NA)
            out[col] = same_cols.bfill(axis=1).iloc[:, 0].fillna("")

    return out


def to_date_string(x: Any) -> str:
    if x is None or x == "":
        return ""
    if isinstance(x, datetime):
        return x.date().isoformat()
    if isinstance(x, date):
        return x.isoformat()

    s = normalize_text(x)
    s = s.replace(".", "/").replace("-", "/")
    s = zenkaku_to_hankaku_digits(s)

    m = re.match(r"^(\d{4})/(\d{1,2})/(\d{1,2})$", s)
    if m:
        y, mm, dd = m.groups()
        return f"{int(y):04d}-{int(mm):02d}-{int(dd):02d}"

    try:
        dt = pd.to_datetime(s)
        return dt.date().isoformat()
    except Exception:
        return normalize_text(x)


def get_year_from_date_string(x: Any) -> Optional[int]:
    if x is None or x == "":
        return None

    if isinstance(x, datetime):
        return x.year
    if isinstance(x, date):
        return x.year

    s = normalize_text(x)
    if s == "":
        return None

    try:
        return int(pd.to_datetime(s).year)
    except Exception:
        m = re.match(r"^(\d{4})", s)
        if m:
            return int(m.group(1))
    return None


def get_available_years(df: pd.DataFrame) -> List[int]:
    df = combine_duplicate_columns(df)
    years = (
        df["日付"]
        .map(get_year_from_date_string)
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    return sorted(years)


def to_int_money(x: Any) -> Optional[int]:
    if x is None or x == "":
        return None

    if isinstance(x, (int, float)) and not pd.isna(x):
        return int(round(float(x)))

    s = normalize_text(x)
    s = zenkaku_to_hankaku_digits(s)
    s = s.replace(",", "").replace("円", "").replace("　", "")
    if s == "":
        return None

    m = re.search(r"-?\d+(\.\d+)?", s)
    if not m:
        return None
    return int(round(float(m.group(0))))


def to_int_distance(x: Any) -> Optional[int]:
    if x is None or x == "":
        return None

    if isinstance(x, (int, float)) and not pd.isna(x):
        return int(round(float(x)))

    s = normalize_text(x)
    s = zenkaku_to_hankaku_digits(s)
    s = s.replace(",", "").replace("　", "")
    if s == "":
        return None

    m = re.search(r"(\d{3,4})", s)
    if not m:
        return None
    return int(m.group(1))


def normalize_age_condition(x: Any) -> str:
    s = normalize_text(x)
    s = zenkaku_to_hankaku_digits(s)
    s = s.replace(" ", "").replace("　", "")

    if s in {"3歳", "３歳"}:
        return "3歳"
    if s in {"3歳以上", "３歳以上"}:
        return "3歳以上"
    if s in {"4歳以上", "４歳以上"}:
        return "4歳以上"

    return normalize_text(x)


def join_marks(marks: Tuple[str, ...]) -> str:
    return "".join(marks)


def format_pattern_brief(selected_patterns: Dict[str, "BuyPattern"]) -> str:
    active = []
    for bet_type in BET_TYPES:
        if bet_type in selected_patterns:
            active.append(bet_type)
    return "/".join(active) if active else "-"


def count_trio_dual_axis_form_points(marks: Tuple[str, ...]) -> int:
    n = len(marks)
    return math.comb(n + 1, 3) - math.comb(n - 1, 3)


def safe_mode_or_first(series: pd.Series):
    s = series.dropna()
    if len(s) == 0:
        return None
    mode = s.mode()
    if len(mode) > 0:
        return mode.iloc[0]
    return s.iloc[0]


def filter_df_by_year(df: pd.DataFrame, target_year: int) -> pd.DataFrame:
    df = combine_duplicate_columns(df)
    years = df["日付"].map(get_year_from_date_string)
    return df[years == int(target_year)].copy()


def unique_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        x_norm = normalize_text(x)
        if x_norm == "" or x_norm in seen:
            continue
        seen.add(x_norm)
        out.append(x_norm)
    return out


def select_rows_by_race_names(ranking_df: pd.DataFrame, race_names: List[str]) -> pd.DataFrame:
    if ranking_df.empty or not race_names:
        return pd.DataFrame(columns=ranking_df.columns if not ranking_df.empty else [])

    ordered_names = unique_preserve_order(race_names)
    order_map = {race_name: i for i, race_name in enumerate(ordered_names)}

    out = ranking_df[ranking_df["レース名"].isin(order_map.keys())].copy()
    if out.empty:
        return out

    out["__priority_order"] = out["レース名"].map(order_map)
    out = out.sort_values(["__priority_order"]).drop(columns="__priority_order").reset_index(drop=True)
    return out


def filter_completed_result_rows(df: pd.DataFrame) -> pd.DataFrame:
    df = combine_duplicate_columns(df)

    required_filter_cols = ["1着", "◎"]
    missing_cols = [col for col in required_filter_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{', '.join(missing_cols)}列が見つからないため、未入力レコードを除外できません。"
        )

    out = df[
        (df["1着"].map(normalize_text) != "") &
        (df["◎"].map(normalize_text) != "")
    ].copy()

    return out.reset_index(drop=True)


# --------------------------------------------
# Numbers -> DataFrame
# --------------------------------------------
def find_target_tables(doc: Document, sheet_keyword: str = "予想"):
    targets = []
    for sheet in doc.sheets:
        if sheet_keyword in sheet.name:
            for table in sheet.tables:
                targets.append((sheet.name, table.name, table))
    return targets


def table_to_dataframe(table) -> pd.DataFrame:
    rows = table.rows(values_only=True)
    if not rows or len(rows) < 2:
        return pd.DataFrame()

    headers = [normalize_text(x) for x in rows[0]]
    df = pd.DataFrame(rows[1:], columns=headers)

    if len(df) > 0:
        df = df[~df.apply(lambda r: all(normalize_text(v) == "" for v in r), axis=1)].copy()

    return df


def standardize_prediction_df(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=REQUIRED_COLUMNS + OPTIONAL_COLUMNS)

    rename_map = {}
    for col in df.columns:
        canon = canonical_column_name(col)
        if canon:
            rename_map[col] = canon

    df = df.rename(columns=rename_map)
    df = combine_duplicate_columns(df)

    for col in REQUIRED_COLUMNS + OPTIONAL_COLUMNS:
        if col not in df.columns:
            df[col] = None

    df = df[REQUIRED_COLUMNS + OPTIONAL_COLUMNS].copy()
    df = combine_duplicate_columns(df)

    df["レース名"] = df["レース名"].map(normalize_text)
    df["日付"] = df["日付"].map(to_date_string)
    df["距離"] = df["距離"].map(to_int_distance)
    df["格"] = df["格"].map(normalize_text)
    df["年齢"] = df["年齢"].map(normalize_age_condition)
    df["単勝"] = df["単勝"].map(to_int_money)
    df["馬連"] = df["馬連"].map(to_int_money)
    df["馬単"] = df["馬単"].map(to_int_money)
    df["３連複"] = df["３連複"].map(to_int_money)
    df["３連単"] = df["３連単"].map(to_int_money)
    df["結果"] = df["結果"].map(normalize_text)
    df["◎"] = df["◎"].map(normalize_text)
    df["競馬場"] = df["競馬場"].map(normalize_text)
    df["1着"] = df["1着"].map(normalize_text)

    df = df[
        (df["レース名"] != "") &
        (df["日付"] != "") &
        df["距離"].notna() &
        (df["格"] != "") &
        (df["年齢"] != "") &
        df["単勝"].notna() &
        df["馬連"].notna() &
        df["馬単"].notna() &
        df["３連複"].notna() &
        df["３連単"].notna() &
        (df["結果"] != "")
    ].copy()

    df["距離"] = df["距離"].astype(int)
    df["単勝"] = df["単勝"].astype(int)
    df["馬連"] = df["馬連"].astype(int)
    df["馬単"] = df["馬単"].astype(int)
    df["３連複"] = df["３連複"].astype(int)
    df["３連単"] = df["３連単"].astype(int)
    df["年齢"] = pd.Categorical(
        df["年齢"],
        categories=["3歳", "3歳以上", "4歳以上"],
        ordered=True,
    )

    df = df.sort_values(["レース名", "日付"]).reset_index(drop=True)
    return df


def extract_from_numbers(numbers_path: str, sheet_keyword: str = "予想") -> pd.DataFrame:
    doc = Document(numbers_path)
    targets = find_target_tables(doc, sheet_keyword=sheet_keyword)

    if not targets:
        raise ValueError(f"'{sheet_keyword}' を含むシートが見つかりませんでした。")

    frames = []
    for _, _, table in targets:
        raw_df = table_to_dataframe(table)
        std_df = standardize_prediction_df(raw_df)
        if not std_df.empty:
            frames.append(std_df)

    if not frames:
        raise ValueError(
            f"'{sheet_keyword}' シート内に、必要列 {REQUIRED_COLUMNS} を持つ表が見つかりませんでした。"
        )

    all_df = pd.concat(frames, ignore_index=True)
    all_df = combine_duplicate_columns(all_df)
    all_df = all_df.drop_duplicates(subset=REQUIRED_COLUMNS + OPTIONAL_COLUMNS).reset_index(drop=True)
    return all_df


# --------------------------------------------
# 買い方定義
# --------------------------------------------
@dataclass(frozen=True)
class BuyPattern:
    key: str
    bet_type: str
    payout_col: str
    marks: Tuple[str, ...]
    point_count: int
    description: str
    hit_rule: str


def make_buy_patterns() -> List[BuyPattern]:
    patterns: List[BuyPattern] = []

    patterns.append(
        BuyPattern(
            key="単勝_◎",
            bet_type="単勝",
            payout_col="単勝",
            marks=tuple(),
            point_count=1,
            description="◎ 1点",
            hit_rule="win",
        )
    )

    for n in (4, 5, 6):
        marks = tuple(RANK_CANDIDATES[:n])

        patterns.append(
            BuyPattern(
                key=f"馬連_◎流し{n}",
                bet_type="馬連",
                payout_col="馬連",
                marks=marks,
                point_count=len(marks),
                description=f"◎-{join_marks(marks)} {len(marks)}点流し",
                hit_rule="pair_anchor",
            )
        )

        patterns.append(
            BuyPattern(
                key=f"馬連_BOX{n}",
                bet_type="馬連",
                payout_col="馬連",
                marks=marks,
                point_count=math.comb(len(marks), 2),
                description=f"{join_marks(marks)} {math.comb(len(marks), 2)}点BOX",
                hit_rule="pair_box",
            )
        )

        patterns.append(
            BuyPattern(
                key=f"馬単_◎1着固定{n}",
                bet_type="馬単",
                payout_col="馬単",
                marks=marks,
                point_count=len(marks),
                description=f"◎→{join_marks(marks)} {len(marks)}点流し",
                hit_rule="umatan_anchor",
            )
        )

        dual_axis_points = count_trio_dual_axis_form_points(marks)
        patterns.append(
            BuyPattern(
                key=f"３連複_◎◯フォーメ{n}",
                bet_type="３連複",
                payout_col="３連複",
                marks=marks,
                point_count=dual_axis_points,
                description=f"◎◯-◎〜{marks[-1]}-◎〜{marks[-1]} {dual_axis_points}点フォーメ",
                hit_rule="trio_dual_axis_form",
            )
        )

        patterns.append(
            BuyPattern(
                key=f"３連複_◎流し{n}",
                bet_type="３連複",
                payout_col="３連複",
                marks=marks,
                point_count=math.comb(len(marks), 2),
                description=f"◎-{join_marks(marks)} {math.comb(len(marks), 2)}点流し",
                hit_rule="trio_anchor",
            )
        )

        patterns.append(
            BuyPattern(
                key=f"３連複_◎抜きBOX{n}",
                bet_type="３連複",
                payout_col="３連複",
                marks=marks,
                point_count=math.comb(len(marks), 3),
                description=f"{join_marks(marks)} {math.comb(len(marks), 3)}点BOX",
                hit_rule="trio_box",
            )
        )

        patterns.append(
            BuyPattern(
                key=f"３連単_◎1着固定{n}",
                bet_type="３連単",
                payout_col="３連単",
                marks=marks,
                point_count=len(marks) * (len(marks) - 1),
                description=f"◎→{join_marks(marks)}→{join_marks(marks)} {len(marks) * (len(marks) - 1)}点流し",
                hit_rule="sanrentan_anchor",
            )
        )

    return patterns


BUY_PATTERNS = make_buy_patterns()
PATTERN_MAP = {p.key: p for p in BUY_PATTERNS}
PATTERNS_BY_TYPE = {
    bet_type: [p for p in BUY_PATTERNS if p.bet_type == bet_type]
    for bet_type in BET_TYPES
}


# --------------------------------------------
# データ構造
# --------------------------------------------
@dataclass
class RaceRecord:
    race_name: str
    date: str
    payouts: Dict[str, int]
    result: str


@dataclass
class SimulationResult:
    race_count: int
    total_stake: int
    total_return: float
    total_profit: float
    roi: float
    hit_count: int
    hit_rate: float
    trimmed_race_count: int
    trimmed_total_stake: int
    trimmed_total_return: float
    trimmed_total_profit: float
    trimmed_roi: float
    trimmed_hit_count: int
    trimmed_hit_rate: float
    trimmed_upper_count: int
    selected_pattern_keys: Dict[str, str]
    selected_pattern_descs: Dict[str, str]
    each_stakes_by_type: Dict[str, int]
    point_counts_by_type: Dict[str, int]
    total_by_type: Dict[str, int]


def df_to_records(df: pd.DataFrame) -> List[RaceRecord]:
    df = combine_duplicate_columns(df)
    records: List[RaceRecord] = []

    for _, row in df.iterrows():
        payouts = {col: int(row[col]) for col in PAYOUT_COLUMNS}
        records.append(
            RaceRecord(
                race_name=str(row["レース名"]),
                date=str(row["日付"]),
                payouts=payouts,
                result=str(row["結果"]),
            )
        )
    return records


def split_result(result: str) -> List[str]:
    s = str(result)
    s = s.replace(">", "＞").replace("○", "◯")
    return [x.strip() for x in s.split("＞") if x.strip()]


def filter_by_race(records: List[RaceRecord], race_name: str) -> List[RaceRecord]:
    return [r for r in records if r.race_name == race_name]


# --------------------------------------------
# 的中判定
# --------------------------------------------
def is_hit_pattern(result: str, pattern: BuyPattern) -> bool:
    order = split_result(result)
    mark_set = set(pattern.marks)

    if pattern.hit_rule == "win":
        return len(order) >= 1 and order[0] == "◎"

    if pattern.hit_rule == "pair_anchor":
        top2 = order[:2]
        return (
            len(top2) == 2
            and "◎" in top2
            and any(x in mark_set for x in top2 if x != "◎")
        )

    if pattern.hit_rule == "pair_box":
        top2 = order[:2]
        return len(top2) == 2 and len(set(top2)) == 2 and all(x in mark_set for x in top2)

    if pattern.hit_rule == "umatan_anchor":
        top2 = order[:2]
        return len(top2) == 2 and top2[0] == "◎" and top2[1] in mark_set

    if pattern.hit_rule == "trio_dual_axis_form":
        top3 = order[:3]
        if len(top3) != 3:
            return False
        allowed = {"◎"} | mark_set
        return (
            len(set(top3)) == 3
            and all(x in allowed for x in top3)
            and any(x in {"◎", "◯"} for x in top3)
        )

    if pattern.hit_rule == "trio_anchor":
        top3 = order[:3]
        if len(top3) < 3 or "◎" not in top3:
            return False
        partners = [x for x in top3 if x != "◎" and x in mark_set]
        return len(partners) >= 2

    if pattern.hit_rule == "trio_box":
        top3 = order[:3]
        return len(top3) == 3 and len(set(top3)) == 3 and all(x in mark_set for x in top3)

    if pattern.hit_rule == "sanrentan_anchor":
        top3 = order[:3]
        return (
            len(top3) == 3
            and top3[0] == "◎"
            and top3[1] in mark_set
            and top3[2] in mark_set
            and top3[1] != top3[2]
        )

    raise ValueError(f"未知の hit_rule: {pattern.hit_rule}")


# --------------------------------------------
# トリム計算
# --------------------------------------------
def get_actual_upper_trim_count(n_items: int, upper_trim_count: int) -> int:
    if n_items <= 1:
        return 0
    return min(max(upper_trim_count, 0), n_items - 1)


def calc_trimmed_summary(
    race_returns: List[Dict[str, float]],
    upper_trim_count: int = 2,
) -> Dict[str, Any]:
    if not race_returns:
        return {
            "trimmed_race_count": 0,
            "trimmed_total_stake": 0,
            "trimmed_total_return": 0.0,
            "trimmed_total_profit": 0.0,
            "trimmed_roi": 0.0,
            "trimmed_hit_count": 0,
            "trimmed_hit_rate": 0.0,
            "trimmed_upper_count": 0,
        }

    sorted_rows = sorted(
        race_returns,
        key=lambda x: (x["profit"], x["date"]),
        reverse=True,
    )
    actual_trim = get_actual_upper_trim_count(len(sorted_rows), upper_trim_count)
    kept = sorted_rows[actual_trim:]

    trimmed_total_stake = sum(x["stake"] for x in kept)
    trimmed_total_return = sum(x["returned"] for x in kept)
    trimmed_total_profit = trimmed_total_return - trimmed_total_stake
    trimmed_roi = trimmed_total_return / trimmed_total_stake if trimmed_total_stake else 0.0
    trimmed_hit_count = sum(1 for x in kept if x["hit"])
    trimmed_hit_rate = trimmed_hit_count / len(kept) if kept else 0.0

    return {
        "trimmed_race_count": len(kept),
        "trimmed_total_stake": trimmed_total_stake,
        "trimmed_total_return": trimmed_total_return,
        "trimmed_total_profit": trimmed_total_profit,
        "trimmed_roi": trimmed_roi,
        "trimmed_hit_count": trimmed_hit_count,
        "trimmed_hit_rate": trimmed_hit_rate,
        "trimmed_upper_count": actual_trim,
    }


def calc_summary_from_race_returns(
    race_returns: List[Dict[str, float]],
    upper_trim_count: int,
) -> Dict[str, Any]:
    total_stake = int(sum(x["stake"] for x in race_returns))
    total_return = float(sum(x["returned"] for x in race_returns))
    total_profit = total_return - total_stake
    roi = total_return / total_stake if total_stake else 0.0
    hit_count = sum(1 for x in race_returns if x["hit"])
    hit_rate = hit_count / len(race_returns) if race_returns else 0.0
    trimmed = calc_trimmed_summary(race_returns, upper_trim_count=upper_trim_count)

    return {
        "race_count": len(race_returns),
        "total_stake": total_stake,
        "total_return": total_return,
        "total_profit": total_profit,
        "roi": roi,
        "hit_count": hit_count,
        "hit_rate": hit_rate,
        **trimmed,
    }


# --------------------------------------------
# シミュレーション
# --------------------------------------------
def calc_race_returns(
    races: List[RaceRecord],
    selected_patterns: Dict[str, BuyPattern],
    each_stakes_by_type: Dict[str, int],
) -> Tuple[List[Dict[str, float]], Dict[str, int], int]:
    total_by_type = {
        bet_type: selected_patterns[bet_type].point_count * each_stakes_by_type[bet_type]
        for bet_type in selected_patterns
    }

    per_race_stake = sum(total_by_type.values())
    if per_race_stake <= 0:
        raise ValueError("1レースあたりの購入金額が0円です。")

    race_returns: List[Dict[str, float]] = []

    for race in races:
        returned = 0.0
        race_hit = False

        for bet_type, pattern in selected_patterns.items():
            each = each_stakes_by_type.get(bet_type, 0)
            if each <= 0:
                continue

            if is_hit_pattern(race.result, pattern):
                returned += race.payouts[pattern.payout_col] * (each / 100)
                race_hit = True

        profit = returned - per_race_stake
        race_returns.append({
            "date": race.date,
            "stake": per_race_stake,
            "returned": returned,
            "profit": profit,
            "hit": race_hit,
        })

    return race_returns, total_by_type, per_race_stake


def simulate(
    races: List[RaceRecord],
    selected_patterns: Dict[str, BuyPattern],
    each_stakes_by_type: Dict[str, int],
    upper_trim_count: int = 2,
) -> SimulationResult:
    if not races:
        raise ValueError("対象レースが0件です。")

    race_returns, total_by_type, _ = calc_race_returns(
        races=races,
        selected_patterns=selected_patterns,
        each_stakes_by_type=each_stakes_by_type,
    )

    summary = calc_summary_from_race_returns(
        race_returns=race_returns,
        upper_trim_count=upper_trim_count,
    )

    return SimulationResult(
        race_count=summary["race_count"],
        total_stake=summary["total_stake"],
        total_return=summary["total_return"],
        total_profit=summary["total_profit"],
        roi=summary["roi"],
        hit_count=summary["hit_count"],
        hit_rate=summary["hit_rate"],
        trimmed_race_count=summary["trimmed_race_count"],
        trimmed_total_stake=summary["trimmed_total_stake"],
        trimmed_total_return=summary["trimmed_total_return"],
        trimmed_total_profit=summary["trimmed_total_profit"],
        trimmed_roi=summary["trimmed_roi"],
        trimmed_hit_count=summary["trimmed_hit_count"],
        trimmed_hit_rate=summary["trimmed_hit_rate"],
        trimmed_upper_count=summary["trimmed_upper_count"],
        selected_pattern_keys={k: v.key for k, v in selected_patterns.items()},
        selected_pattern_descs={k: v.description for k, v in selected_patterns.items()},
        each_stakes_by_type=dict(each_stakes_by_type),
        point_counts_by_type={k: v.point_count for k, v in selected_patterns.items()},
        total_by_type=total_by_type,
    )


def _roi_score(x: SimulationResult) -> Tuple[float, float, float, float, float]:
    return (
        x.trimmed_roi,
        x.trimmed_hit_rate,
        x.trimmed_total_profit,
        x.roi,
        x.total_profit,
    )


def _profit_score(x: SimulationResult) -> Tuple[float, float, float, float, float]:
    return (
        x.trimmed_total_profit,
        x.trimmed_roi,
        x.trimmed_hit_rate,
        x.total_profit,
        x.roi,
    )


def build_pattern_summary_from_result(result: SimulationResult) -> str:
    parts = []

    for bet_type in BET_TYPES:
        pattern_key = result.selected_pattern_keys.get(bet_type, "")
        each = int(result.each_stakes_by_type.get(bet_type, 0) or 0)
        point_count = int(result.point_counts_by_type.get(bet_type, 0) or 0)
        total = int(result.total_by_type.get(bet_type, 0) or 0)

        if pattern_key == "" or each <= 0:
            continue

        parts.append(
            f"{bet_type}: {pattern_key} / 1点{each:,}円 / {point_count}点 / 計{total:,}円"
        )

    return " | ".join(parts)


def search_allocations(
    races: List[RaceRecord],
    max_budget_per_race: int,
    step: int = 100,
    upper_trim_count: int = 2,
    race_name: str = "",
    show_progress: bool = False,
    progress_interval_sec: float = 0.5,
) -> Tuple[Optional[SimulationResult], Optional[SimulationResult]]:
    if not races:
        return None, None

    if step <= 0:
        raise ValueError("step は正の整数である必要があります。")

    max_units = max_budget_per_race // step
    best_roi: Optional[SimulationResult] = None
    best_profit: Optional[SimulationResult] = None

    pattern_choices = []
    for bet_type in BET_TYPES:
        pattern_choices.append([None] + PATTERNS_BY_TYPE[bet_type])

    eval_count = 0
    start_time = time.time()
    last_progress_time = 0.0

    def print_progress(current_patterns: Dict[str, BuyPattern], force: bool = False) -> None:
        nonlocal last_progress_time

        if not show_progress:
            return

        now = time.time()
        if not force and (now - last_progress_time) < progress_interval_sec:
            return

        elapsed = now - start_time
        current_state = format_pattern_brief(current_patterns)

        best_roi_text = f"{best_roi.trimmed_roi:.3f}" if best_roi is not None else "-"
        best_profit_text = (
            f"{int(round(best_profit.trimmed_total_profit)):+,}円"
            if best_profit is not None else "-"
        )

        race_label = race_name if race_name else "集計中"
        msg = (
            f"\r[{race_label}] {eval_count:,}件"
            f" / bestROI {best_roi_text}"
            f" / best収支 {best_profit_text}"
            f" / 現在 {current_state}"
            f" / {elapsed:.1f}s"
        )

        sys.stdout.write(msg)
        sys.stdout.flush()
        last_progress_time = now

    for choice_tuple in product(*pattern_choices):
        selected_patterns = {
            p.bet_type: p for p in choice_tuple if p is not None
        }

        if not selected_patterns:
            continue

        active_bet_types = list(selected_patterns.keys())
        active_patterns = [selected_patterns[bt] for bt in active_bet_types]

        min_units_list = [
            math.ceil(MIN_EACH_BY_TYPE[p.bet_type] / step)
            for p in active_patterns
        ]
        weights = [p.point_count for p in active_patterns]

        min_required_units = sum(w * u for w, u in zip(weights, min_units_list))
        if min_required_units > max_units:
            continue

        suffix_min_required = [0] * (len(active_patterns) + 1)
        for i in range(len(active_patterns) - 1, -1, -1):
            suffix_min_required[i] = suffix_min_required[i + 1] + weights[i] * min_units_list[i]

        current_each_stakes: Dict[str, int] = {}

        def dfs(idx: int, remaining_units: int) -> None:
            nonlocal best_roi, best_profit, eval_count

            if idx == len(active_patterns):
                summary = simulate(
                    races=races,
                    selected_patterns=selected_patterns,
                    each_stakes_by_type=current_each_stakes,
                    upper_trim_count=upper_trim_count,
                )

                eval_count += 1

                if best_roi is None or _roi_score(summary) > _roi_score(best_roi):
                    best_roi = summary
                if best_profit is None or _profit_score(summary) > _profit_score(best_profit):
                    best_profit = summary

                print_progress(selected_patterns)
                return

            pattern = active_patterns[idx]
            weight = pattern.point_count
            min_units = min_units_list[idx]

            max_each_units = remaining_units // weight
            for u in range(min_units, max_each_units + 1):
                rem = remaining_units - weight * u
                if rem < suffix_min_required[idx + 1]:
                    continue

                current_each_stakes[pattern.bet_type] = u * step
                dfs(idx + 1, rem)

            current_each_stakes.pop(pattern.bet_type, None)

        dfs(0, max_units)

    print_progress({}, force=True)
    if show_progress:
        sys.stdout.write("\n")
        sys.stdout.flush()

    return best_roi, best_profit


def scale_roi_result_to_target_budget(
    races: List[RaceRecord],
    result: Optional[SimulationResult],
    upper_trim_count: int,
    target_budget_per_race: int = 10000,
) -> Optional[SimulationResult]:
    if result is None:
        return None

    if result.race_count <= 0:
        return result

    current_budget = result.total_stake // result.race_count
    if current_budget <= 0:
        return result

    if current_budget >= target_budget_per_race:
        return result

    multiplier = target_budget_per_race // current_budget
    if multiplier <= 1:
        return result

    selected_patterns: Dict[str, BuyPattern] = {}
    scaled_each_stakes_by_type: Dict[str, int] = {}

    for bet_type in BET_TYPES:
        pattern_key = result.selected_pattern_keys.get(bet_type, "")
        each = int(result.each_stakes_by_type.get(bet_type, 0) or 0)

        if pattern_key == "" or each <= 0:
            continue

        pattern = PATTERN_MAP.get(pattern_key)
        if pattern is None:
            continue

        selected_patterns[bet_type] = pattern
        scaled_each_stakes_by_type[bet_type] = each * multiplier

    if not selected_patterns:
        return result

    return simulate(
        races=races,
        selected_patterns=selected_patterns,
        each_stakes_by_type=scaled_each_stakes_by_type,
        upper_trim_count=upper_trim_count,
    )


# --------------------------------------------
# 行生成・レース別ランキング
# --------------------------------------------
def build_result_row(
    label_name: str,
    label_value: str,
    best_roi: SimulationResult,
    best_profit: Optional[SimulationResult],
) -> Dict[str, Any]:
    row = {
        label_name: label_value,
        "件数": best_roi.race_count,
        "トリム後件数": best_roi.trimmed_race_count,
        "1レース予算": best_roi.total_stake // best_roi.race_count if best_roi.race_count else 0,
        "トリム後ROI": round(best_roi.trimmed_roi, 6),
        "トリム後的中率": round(best_roi.trimmed_hit_rate, 6),
        "通常ROI": round(best_roi.roi, 6),
        "通常的中率": round(best_roi.hit_rate, 6),
        "トリム後収支": int(round(best_roi.trimmed_total_profit)),
        "通常収支": int(round(best_roi.total_profit)),
        "最良収支_トリム後収支": int(round(best_profit.trimmed_total_profit)) if best_profit else None,
        "最良収支_トリム後ROI": round(best_profit.trimmed_roi, 6) if best_profit else None,
        "最良収支_トリム後的中率": round(best_profit.trimmed_hit_rate, 6) if best_profit else None,
    }

    for bet_type in BET_TYPES:
        row[f"{bet_type}_型"] = best_roi.selected_pattern_keys.get(bet_type, "")
        row[f"{bet_type}_1点"] = best_roi.each_stakes_by_type.get(bet_type, 0)
        row[f"{bet_type}_点数"] = best_roi.point_counts_by_type.get(bet_type, 0)
        row[f"{bet_type}_合計"] = best_roi.total_by_type.get(bet_type, 0)

    row["買い方サマリー"] = build_pattern_summary_from_result(best_roi)
    return row


def build_race_row(args):
    race_name_each, target_races, max_budget_per_race, step, upper_trim_count, roi_target_budget_per_race = args

    best_roi, best_profit = search_allocations(
        races=target_races,
        max_budget_per_race=max_budget_per_race,
        step=step,
        upper_trim_count=upper_trim_count,
        race_name=race_name_each,
        show_progress=False,
        progress_interval_sec=1.0,
    )

    if best_roi is None:
        return None

    best_roi = scale_roi_result_to_target_budget(
        races=target_races,
        result=best_roi,
        upper_trim_count=upper_trim_count,
        target_budget_per_race=roi_target_budget_per_race,
    )

    return build_result_row(
        label_name="レース名",
        label_value=race_name_each,
        best_roi=best_roi,
        best_profit=best_profit,
    )


def build_race_jobs(
    all_df: pd.DataFrame,
    max_budget_per_race: int,
    step: int,
    upper_trim_count: int,
    roi_target_budget_per_race: int,
    target_race_names: Optional[List[str]] = None,
    sort_race_names: bool = True,
) -> List[Tuple[str, List[RaceRecord], int, int, int, int]]:
    all_df = combine_duplicate_columns(all_df)
    records = df_to_records(all_df[REQUIRED_COLUMNS])

    if target_race_names is None:
        race_names = list(all_df["レース名"].dropna().astype(str).unique())
        if sort_race_names:
            race_names = sorted(race_names)
    else:
        race_names = unique_preserve_order(target_race_names)

    race_jobs = []
    for race_name_each in race_names:
        target_races = filter_by_race(records, race_name_each)
        if target_races:
            race_jobs.append(
                (
                    race_name_each,
                    target_races,
                    max_budget_per_race,
                    step,
                    upper_trim_count,
                    roi_target_budget_per_race,
                )
            )

    return race_jobs


def build_rows_for_race_jobs(
    race_jobs: List[Tuple[str, List[RaceRecord], int, int, int, int]],
    show_progress: bool = False,
    progress_interval_sec: float = 0.5,
    parallel: bool = False,
    max_workers: Optional[int] = None,
    show_parallel_progress: bool = True,
) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []

    if not race_jobs:
        return rows

    if parallel:
        worker_count = max_workers or os.cpu_count() or 2

        with ProcessPoolExecutor(max_workers=worker_count) as executor:
            futures = [executor.submit(build_race_row, job) for job in race_jobs]

            done_count = 0
            total_count = len(futures)

            for future in as_completed(futures):
                row = future.result()
                done_count += 1

                if row is not None:
                    rows.append(row)

                if show_parallel_progress:
                    msg = f"\r並列集計: {done_count}/{total_count} レース完了"
                    sys.stdout.write(msg)
                    sys.stdout.flush()

        if show_parallel_progress:
            sys.stdout.write("\n")
            sys.stdout.flush()

    else:
        for race_name_each, target_races, max_budget_each, step_each, trim_each, roi_target_budget_each in race_jobs:
            best_roi, best_profit = search_allocations(
                races=target_races,
                max_budget_per_race=max_budget_each,
                step=step_each,
                upper_trim_count=trim_each,
                race_name=race_name_each,
                show_progress=show_progress,
                progress_interval_sec=progress_interval_sec,
            )

            if best_roi is None:
                continue

            best_roi = scale_roi_result_to_target_budget(
                races=target_races,
                result=best_roi,
                upper_trim_count=trim_each,
                target_budget_per_race=roi_target_budget_each,
            )

            row = build_result_row(
                label_name="レース名",
                label_value=race_name_each,
                best_roi=best_roi,
                best_profit=best_profit,
            )
            rows.append(row)

    return rows


def build_all_races_ranking(
    all_df: pd.DataFrame,
    max_budget_per_race: int,
    step: int,
    upper_trim_count: int,
    roi_target_budget_per_race: int,
    show_progress: bool = False,
    progress_interval_sec: float = 0.5,
    parallel: bool = False,
    max_workers: Optional[int] = None,
    show_parallel_progress: bool = True,
) -> pd.DataFrame:
    race_jobs = build_race_jobs(
        all_df=all_df,
        max_budget_per_race=max_budget_per_race,
        step=step,
        upper_trim_count=upper_trim_count,
        roi_target_budget_per_race=roi_target_budget_per_race,
        target_race_names=None,
        sort_race_names=True,
    )

    rows = build_rows_for_race_jobs(
        race_jobs=race_jobs,
        show_progress=show_progress,
        progress_interval_sec=progress_interval_sec,
        parallel=parallel,
        max_workers=max_workers,
        show_parallel_progress=show_parallel_progress,
    )

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = df.sort_values(
        ["トリム後ROI", "トリム後的中率", "トリム後収支", "通常ROI", "通常収支"],
        ascending=False
    ).reset_index(drop=True)
    return df


def build_priority_races_preview(
    all_df: pd.DataFrame,
    priority_race_names: List[str],
    max_budget_per_race: int,
    step: int,
    upper_trim_count: int,
    roi_target_budget_per_race: int,
    show_progress: bool = False,
    progress_interval_sec: float = 0.5,
    parallel: bool = False,
    max_workers: Optional[int] = None,
    show_parallel_progress: bool = True,
) -> pd.DataFrame:
    ordered_names = unique_preserve_order(priority_race_names)
    if not ordered_names:
        return pd.DataFrame()

    race_jobs = build_race_jobs(
        all_df=all_df,
        max_budget_per_race=max_budget_per_race,
        step=step,
        upper_trim_count=upper_trim_count,
        roi_target_budget_per_race=roi_target_budget_per_race,
        target_race_names=ordered_names,
        sort_race_names=False,
    )

    rows = build_rows_for_race_jobs(
        race_jobs=race_jobs,
        show_progress=show_progress,
        progress_interval_sec=progress_interval_sec,
        parallel=parallel,
        max_workers=max_workers,
        show_parallel_progress=show_parallel_progress,
    )

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    df = select_rows_by_race_names(df, ordered_names)
    return df


# --------------------------------------------
# 注力レース抽出
# --------------------------------------------
def add_focus_flag_to_all_races_ranking(
    all_races_ranking_df: pd.DataFrame,
    min_trimmed_roi: float = 1.5,
    min_trimmed_hit_rate: float = 0.30,
) -> pd.DataFrame:
    """
    全レースランキングに注力レース判定列を追加する。

    注力レース定義:
    - トリム後ROI >= min_trimmed_roi
    - トリム後的中率 > min_trimmed_hit_rate
    """
    if all_races_ranking_df.empty:
        return all_races_ranking_df.copy()

    required_cols = ["レース名", "トリム後ROI", "トリム後的中率"]
    missing_cols = [c for c in required_cols if c not in all_races_ranking_df.columns]
    if missing_cols:
        raise ValueError(f"注力レース判定に必要な列がありません: {missing_cols}")

    out = all_races_ranking_df.copy()
    trimmed_roi = pd.to_numeric(out["トリム後ROI"], errors="coerce")
    trimmed_hit_rate = pd.to_numeric(out["トリム後的中率"], errors="coerce")

    is_focus = (
        (trimmed_roi >= float(min_trimmed_roi)) &
        (trimmed_hit_rate > float(min_trimmed_hit_rate))
    )

    out["注力レース"] = is_focus.map({True: "注力", False: "非注力"})

    return out


def build_focus_race_ranking(
    all_races_ranking_df: pd.DataFrame,
    min_trimmed_roi: float = 1.5,
    min_trimmed_hit_rate: float = 0.30,
) -> pd.DataFrame:
    """
    注力レースだけを内部集計用に抽出する。
    表示用の「注力レース一覧」は作らず、
    全レースランキングの「注力レース」列で確認する。
    """
    if all_races_ranking_df.empty:
        return pd.DataFrame(columns=all_races_ranking_df.columns)

    work_df = add_focus_flag_to_all_races_ranking(
        all_races_ranking_df=all_races_ranking_df,
        min_trimmed_roi=min_trimmed_roi,
        min_trimmed_hit_rate=min_trimmed_hit_rate,
    )

    focus_df = work_df[work_df["注力レース"] == "注力"].copy()

    if focus_df.empty:
        return focus_df.reset_index(drop=True)

    focus_df = focus_df.sort_values(
        ["トリム後ROI", "トリム後的中率", "トリム後収支", "通常ROI", "通常収支"],
        ascending=False
    ).reset_index(drop=True)

    return focus_df

def filter_all_df_to_ranking_races(
    all_df: pd.DataFrame,
    ranking_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    ranking_dfに含まれるレース名だけにall_dfを絞る。
    注力レース以降の集計用。
    """
    all_df = combine_duplicate_columns(all_df)

    if all_df.empty or ranking_df.empty or "レース名" not in ranking_df.columns:
        return pd.DataFrame(columns=all_df.columns)

    race_names = set(ranking_df["レース名"].map(normalize_text))
    out = all_df[all_df["レース名"].map(normalize_text).isin(race_names)].copy()

    return out.reset_index(drop=True)


# --------------------------------------------
# レース別最適買い方を使った集計
# --------------------------------------------
def ranking_row_to_patterns_and_stakes(row: pd.Series) -> Tuple[Dict[str, BuyPattern], Dict[str, int]]:
    selected_patterns: Dict[str, BuyPattern] = {}
    each_stakes_by_type: Dict[str, int] = {}

    for bet_type in BET_TYPES:
        pattern_key = normalize_text(row.get(f"{bet_type}_型", ""))
        each = int(row.get(f"{bet_type}_1点", 0) or 0)

        if pattern_key == "" or each <= 0:
            continue

        pattern = PATTERN_MAP.get(pattern_key)
        if pattern is None:
            raise ValueError(f"未知の買い方キーです: {pattern_key}")

        selected_patterns[bet_type] = pattern
        each_stakes_by_type[bet_type] = each

    return selected_patterns, each_stakes_by_type


def make_ranking_row_by_race_name(ranking_df: pd.DataFrame) -> Dict[str, pd.Series]:
    if ranking_df.empty or "レース名" not in ranking_df.columns:
        return {}

    out: Dict[str, pd.Series] = {}
    for _, row in ranking_df.iterrows():
        race_name = normalize_text(row.get("レース名", ""))
        if race_name and race_name not in out:
            out[race_name] = row
    return out


def collect_race_returns_using_race_best_patterns(
    target_df: pd.DataFrame,
    race_ranking_df: pd.DataFrame,
) -> Tuple[List[Dict[str, float]], List[str], List[str]]:
    target_df = combine_duplicate_columns(target_df)
    if target_df.empty or race_ranking_df.empty:
        return [], [], []

    ranking_by_race = make_ranking_row_by_race_name(race_ranking_df)
    race_returns: List[Dict[str, float]] = []
    used_race_names: List[str] = []
    skipped_race_names: List[str] = []

    for race_name, race_df in target_df.groupby("レース名", sort=True):
        race_name_norm = normalize_text(race_name)
        ranking_row = ranking_by_race.get(race_name_norm)

        if ranking_row is None:
            skipped_race_names.append(race_name_norm)
            continue

        selected_patterns, each_stakes_by_type = ranking_row_to_patterns_and_stakes(ranking_row)
        if not selected_patterns:
            skipped_race_names.append(race_name_norm)
            continue

        target_records = df_to_records(race_df[REQUIRED_COLUMNS])
        returns, _, _ = calc_race_returns(
            races=target_records,
            selected_patterns=selected_patterns,
            each_stakes_by_type=each_stakes_by_type,
        )

        race_returns.extend(returns)
        used_race_names.append(race_name_norm)

    return race_returns, unique_preserve_order(used_race_names), unique_preserve_order(skipped_race_names)


def build_group_roi_using_race_best_patterns(
    all_df: pd.DataFrame,
    race_ranking_df: pd.DataFrame,
    group_col: str,
    upper_trim_count: int,
) -> pd.DataFrame:
    all_df = combine_duplicate_columns(all_df)

    if all_df.empty or race_ranking_df.empty or group_col not in all_df.columns:
        return pd.DataFrame()

    work_df = all_df.copy()
    work_df[group_col] = work_df[group_col].map(normalize_text)
    work_df = work_df[work_df[group_col] != ""].copy()

    if work_df.empty:
        return pd.DataFrame()

    rows = []
    group_values = sorted(work_df[group_col].dropna().astype(str).unique())

    for group_value in group_values:
        target_df = work_df[work_df[group_col] == group_value].copy()
        race_returns, used_races, skipped_races = collect_race_returns_using_race_best_patterns(
            target_df=target_df,
            race_ranking_df=race_ranking_df,
        )

        summary = calc_summary_from_race_returns(
            race_returns=race_returns,
            upper_trim_count=upper_trim_count,
        )

        rows.append({
            "集計軸": group_col,
            "集計値": group_value,
            "件数": summary["race_count"],
            "トリム後件数": summary["trimmed_race_count"],
            "使用レース種類数": len(used_races),
            "未使用レース種類数": len(skipped_races),
            "平均1レース投資": int(round(summary["total_stake"] / summary["race_count"])) if summary["race_count"] else 0,
            "トリム後ROI": round(summary["trimmed_roi"], 6),
            "トリム後的中率": round(summary["trimmed_hit_rate"], 6),
            "通常ROI": round(summary["roi"], 6),
            "通常的中率": round(summary["hit_rate"], 6),
            "トリム後収支": int(round(summary["trimmed_total_profit"])),
            "通常収支": int(round(summary["total_profit"])),
            "総投資": int(summary["total_stake"]),
            "総払戻": int(round(summary["total_return"])),
            "買い方ルール": "各レース名ごとの最適買い方を使用",
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.sort_values(
        ["トリム後ROI", "トリム後的中率", "トリム後収支", "通常ROI", "通常収支"],
        ascending=False
    ).reset_index(drop=True)
    return df


def build_grade_year_roi_using_race_best_patterns(
    filtered_all_df: pd.DataFrame,
    race_ranking_df: pd.DataFrame,
    target_years: Optional[List[int]],
    upper_trim_count: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if filtered_all_df.empty or race_ranking_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    work_df = combine_duplicate_columns(filtered_all_df).copy()
    work_df["格"] = work_df["格"].map(normalize_text)
    dates = pd.to_datetime(work_df["日付"], errors="coerce")

    available_years = get_available_years(work_df)
    if target_years is None:
        years = available_years
    else:
        available_years_set = set(available_years)
        years = [int(y) for y in target_years if int(y) in available_years_set]

    grades = sorted(work_df["格"].dropna().astype(str).unique())

    table_rows = []
    detail_rows = []

    for grade in grades:
        table_row = {"格": grade}

        for year in years:
            target_df = work_df[
                (work_df["格"] == grade) &
                (dates.dt.year == int(year))
            ].copy()

            race_returns, used_races, skipped_races = collect_race_returns_using_race_best_patterns(
                target_df=target_df,
                race_ranking_df=race_ranking_df,
            )

            summary = calc_summary_from_race_returns(
                race_returns=race_returns,
                upper_trim_count=upper_trim_count,
            )

            table_row[f"{year}年"] = (
                round(summary["trimmed_roi"], 6)
                if summary["trimmed_total_stake"] else None
            )

            detail_rows.append({
                "格": grade,
                "年": year,
                "件数": summary["race_count"],
                "トリム後件数": summary["trimmed_race_count"],
                "使用レース種類数": len(used_races),
                "未使用レース種類数": len(skipped_races),
                "総投資": int(summary["total_stake"]),
                "総払戻": int(round(summary["total_return"])),
                "総収支": int(round(summary["total_profit"])),
                "ROI": round(summary["roi"], 6),
                "的中率": round(summary["hit_rate"], 6),
                "トリム後ROI": round(summary["trimmed_roi"], 6),
                "トリム後的中率": round(summary["trimmed_hit_rate"], 6),
                "買い方ルール": "各レース名ごとの最適買い方を使用",
            })

        table_rows.append(table_row)

    table_df = pd.DataFrame(table_rows)
    detail_df = pd.DataFrame(detail_rows)

    return table_df, detail_df


def add_grade_total_column_to_grade_year_roi(
    grade_year_roi_df: pd.DataFrame,
    grade_roi_summary_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    格ごとROI集計を、格×年別ROI表の「総計」列として統合する。
    """
    if grade_year_roi_df.empty:
        return grade_year_roi_df

    out = grade_year_roi_df.copy()

    if grade_roi_summary_df.empty:
        out["総計"] = None
        return out

    required_cols = ["集計値", "トリム後ROI"]
    missing_cols = [c for c in required_cols if c not in grade_roi_summary_df.columns]
    if missing_cols:
        out["総計"] = None
        return out

    total_roi_by_grade = (
        grade_roi_summary_df
        .set_index("集計値")["トリム後ROI"]
        .to_dict()
    )

    out["総計"] = out["格"].map(total_roi_by_grade)

    ordered_cols = ["格"] + [c for c in out.columns if c not in {"格", "総計"}] + ["総計"]
    out = out[ordered_cols]

    return out


# --------------------------------------------
# 年間収支計算
# --------------------------------------------
def calc_yearly_profit_using_best_roi_patterns(
    filtered_all_df: pd.DataFrame,
    ranking_df: pd.DataFrame,
    target_year: int,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    year_df = filter_df_by_year(filtered_all_df, target_year)

    if year_df.empty or ranking_df.empty:
        summary = {
            "year": target_year,
            "race_count": 0,
            "total_stake": 0,
            "total_return": 0.0,
            "total_profit": 0.0,
            "roi": 0.0,
            "race_type_count": 0,
        }
        return summary, pd.DataFrame()

    year_records = df_to_records(year_df[REQUIRED_COLUMNS])

    records_by_race: Dict[str, List[RaceRecord]] = {}
    for record in year_records:
        records_by_race.setdefault(record.race_name, []).append(record)

    detail_rows = []
    total_stake = 0
    total_return = 0.0
    race_count = 0

    for _, ranking_row in ranking_df.iterrows():
        race_name = str(ranking_row["レース名"])

        target_races = records_by_race.get(race_name, [])
        if not target_races:
            continue

        selected_patterns, each_stakes_by_type = ranking_row_to_patterns_and_stakes(ranking_row)
        if not selected_patterns:
            continue

        sim = simulate(
            races=target_races,
            selected_patterns=selected_patterns,
            each_stakes_by_type=each_stakes_by_type,
            upper_trim_count=0,
        )

        total_stake += sim.total_stake
        total_return += sim.total_return
        race_count += sim.race_count

        detail_rows.append({
            "年": target_year,
            "レース名": race_name,
            "件数": sim.race_count,
            "的中数": sim.hit_count,
            "的中率": round(sim.hit_rate, 6),
            "総投資": int(sim.total_stake),
            "総払戻": int(round(sim.total_return)),
            "総収支": int(round(sim.total_profit)),
            "ROI": round(sim.roi, 6),
            "買い方サマリー": build_pattern_summary_from_result(sim),
        })

    detail_df = pd.DataFrame(detail_rows)
    if not detail_df.empty:
        detail_df = detail_df.sort_values(
            ["総収支", "ROI", "的中率", "件数"],
            ascending=False
        ).reset_index(drop=True)

    summary = {
        "year": target_year,
        "race_count": race_count,
        "total_stake": int(total_stake),
        "total_return": float(total_return),
        "total_profit": float(total_return - total_stake),
        "roi": (float(total_return) / total_stake) if total_stake else 0.0,
        "race_type_count": len(detail_df),
    }
    return summary, detail_df


def calc_all_yearly_profit_using_best_roi_patterns(
    raw_all_df: pd.DataFrame,
    filtered_all_df: pd.DataFrame,
    ranking_df: pd.DataFrame,
    target_years: Optional[List[int]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if ranking_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    raw_all_df = combine_duplicate_columns(raw_all_df)
    available_years = get_available_years(raw_all_df)

    if target_years is None:
        years = available_years
    else:
        available_years_set = set(available_years)
        years = [int(y) for y in target_years if int(y) in available_years_set]

    summary_rows = []
    detail_frames = []

    for year in years:
        yearly_summary, yearly_detail_df = calc_yearly_profit_using_best_roi_patterns(
            filtered_all_df=filtered_all_df,
            ranking_df=ranking_df,
            target_year=year,
        )

        summary_rows.append({
            "年": yearly_summary["year"],
            "対象レース種類数": yearly_summary["race_type_count"],
            "対象レース件数": yearly_summary["race_count"],
            "総投資": int(yearly_summary["total_stake"]),
            "総払戻": int(round(yearly_summary["total_return"])),
            "総収支": int(round(yearly_summary["total_profit"])),
            "年間ROI": round(yearly_summary["roi"], 6),
        })

        if not yearly_detail_df.empty:
            detail_frames.append(yearly_detail_df.copy())

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty:
        summary_df = summary_df.sort_values("年").reset_index(drop=True)

    all_detail_df = pd.concat(detail_frames, ignore_index=True) if detail_frames else pd.DataFrame()
    if not all_detail_df.empty:
        all_detail_df = all_detail_df.sort_values(
            ["年", "総収支", "ROI"],
            ascending=[True, False, False]
        ).reset_index(drop=True)

    return summary_df, all_detail_df


# --------------------------------------------
# 月別累計トリム後ROI
# --------------------------------------------
def calc_cumulative_trimmed_roi_using_best_roi_patterns(
    filtered_all_df: pd.DataFrame,
    ranking_df: pd.DataFrame,
    target_year: int,
    target_month: int,
    upper_trim_count: int,
) -> Dict[str, Any]:
    if ranking_df.empty:
        return {
            "年": target_year,
            "月": target_month,
            "件数": 0,
            "トリム後件数": 0,
            "トリム後ROI": 0.0,
        }

    filtered_all_df = combine_duplicate_columns(filtered_all_df)
    dates = pd.to_datetime(filtered_all_df["日付"], errors="coerce")
    period_df = filtered_all_df[
        (dates.dt.year == int(target_year)) &
        (dates.dt.month <= int(target_month))
    ].copy()

    if period_df.empty:
        return {
            "年": target_year,
            "月": target_month,
            "件数": 0,
            "トリム後件数": 0,
            "トリム後ROI": 0.0,
        }

    race_returns, _, _ = collect_race_returns_using_race_best_patterns(
        target_df=period_df,
        race_ranking_df=ranking_df,
    )

    trimmed = calc_trimmed_summary(race_returns, upper_trim_count=upper_trim_count)

    return {
        "年": target_year,
        "月": target_month,
        "件数": len(race_returns),
        "トリム後件数": trimmed["trimmed_race_count"],
        "トリム後ROI": round(trimmed["trimmed_roi"], 6),
    }


def build_monthly_cumulative_trimmed_roi_table(
    raw_all_df: pd.DataFrame,
    filtered_all_df: pd.DataFrame,
    ranking_df: pd.DataFrame,
    target_years: Optional[List[int]],
    target_months: List[int],
    upper_trim_count: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    raw_all_df = combine_duplicate_columns(raw_all_df)
    available_years = get_available_years(raw_all_df)

    if target_years is None:
        years = available_years
    else:
        available_years_set = set(available_years)
        years = [int(y) for y in target_years if int(y) in available_years_set]

    months = [int(m) for m in target_months if 1 <= int(m) <= 12]

    detail_rows = []

    for year in years:
        for month in months:
            detail_rows.append(
                calc_cumulative_trimmed_roi_using_best_roi_patterns(
                    filtered_all_df=filtered_all_df,
                    ranking_df=ranking_df,
                    target_year=year,
                    target_month=month,
                    upper_trim_count=upper_trim_count,
                )
            )

    detail_df = pd.DataFrame(detail_rows)

    table_rows = []
    for month in months:
        row = {"月": f"{month}月累計"}

        for year in years:
            matched = detail_df[
                (detail_df["年"] == year) &
                (detail_df["月"] == month)
            ]
            row[f"{year}年"] = (
                float(matched["トリム後ROI"].iloc[0])
                if not matched.empty else 0.0
            )

        table_rows.append(row)

    table_df = pd.DataFrame(table_rows)
    return table_df, detail_df


def build_monthly_cumulative_trimmed_roi_range_label_table(
    monthly_cumulative_trimmed_roi_df: pd.DataFrame,
) -> pd.DataFrame:
    if monthly_cumulative_trimmed_roi_df.empty:
        return pd.DataFrame()

    out = monthly_cumulative_trimmed_roi_df.copy()

    labels = []
    for value in out["月"]:
        s = normalize_text(value)
        m = re.search(r"(\d+)", s)
        month = int(m.group(1)) if m else len(labels) + 1

        if month == 1:
            labels.append("1月累計")
        else:
            labels.append(f"1-{month}月累計")

    out["月"] = labels
    return out


def build_monthly_cumulative_trimmed_roi_print_table(
    monthly_cumulative_trimmed_roi_df: pd.DataFrame,
    yearly_summary_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """
    月別累計ROI表に、年別年間収支/ROIを同じヘッダの総計行として追加する。
    表示用なので、ROIセルと総計セルは文字列化する。
    """
    if monthly_cumulative_trimmed_roi_df.empty:
        return pd.DataFrame()

    out = monthly_cumulative_trimmed_roi_df.copy()
    year_cols = [c for c in out.columns if c != "月"]

    for col in year_cols:
        out[col] = out[col].map(
            lambda v: "-" if pd.isna(v) else f"{float(v):.3f}"
        )

    if yearly_summary_df is not None and not yearly_summary_df.empty:
        yearly_by_col = {
            f"{int(row['年'])}年": row
            for _, row in yearly_summary_df.iterrows()
        }

        total_row = {"月": "年間収支/ROI"}

        for col in year_cols:
            yearly_row = yearly_by_col.get(col)
            if yearly_row is None:
                total_row[col] = "-"
            else:
                total_row[col] = (
                    f"{int(yearly_row['総収支']):,}円"
                    f" / ROI {float(yearly_row['年間ROI']):.3f}"
                )

        out = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)

    return out


# --------------------------------------------
# 図・表示
# --------------------------------------------
def build_plot_meta_df(all_df: pd.DataFrame) -> pd.DataFrame:
    all_df = combine_duplicate_columns(all_df)

    plot_meta_df = (
        all_df.groupby("レース名", as_index=False)
        .agg(
            距離=("距離", safe_mode_or_first),
            格=("格", safe_mode_or_first),
            年齢=("年齢", safe_mode_or_first),
        )
    )

    plot_meta_df["年齢"] = pd.Categorical(
        plot_meta_df["年齢"],
        categories=["3歳", "3歳以上", "4歳以上"],
        ordered=True,
    )
    return plot_meta_df


def build_scatter_plot(
    ranking_df: pd.DataFrame,
    all_df: pd.DataFrame,
    html_path: str,
    source_csv_path: str,
):
    if ranking_df.empty:
        return None, pd.DataFrame()

    plot_meta_df = build_plot_meta_df(all_df)

    plot_df = ranking_df.merge(plot_meta_df, on="レース名", how="left")
    plot_df = plot_df.dropna(subset=["距離", "格", "年齢"]).copy()

    if plot_df.empty:
        return None, plot_df

    plot_df["距離"] = plot_df["距離"].astype(int)
    plot_df["年齢"] = pd.Categorical(
        plot_df["年齢"],
        categories=["3歳", "3歳以上", "4歳以上"],
        ordered=True,
    )

    if "注力レース" not in plot_df.columns:
        plot_df = add_focus_flag_to_all_races_ranking(
            all_races_ranking_df=plot_df,
            min_trimmed_roi=FOCUS_TRIMMED_ROI_MIN,
            min_trimmed_hit_rate=FOCUS_TRIMMED_HIT_RATE_MIN,
        )

    # 注力レースは格ごとに色分け、非注力レースは表示したままグレーにする
    plot_df["グラフ色区分"] = plot_df["格"].astype(str)
    plot_df.loc[plot_df["注力レース"] != "注力", "グラフ色区分"] = "非注力"

    plot_df.to_csv(source_csv_path, index=False, encoding="utf-8-sig")

    fig = px.scatter(
        plot_df,
        x="距離",
        y="トリム後ROI",
        color="グラフ色区分",
        symbol="年齢",
        category_orders={
            "年齢": ["3歳", "3歳以上", "4歳以上"],
        },
        color_discrete_map={
            "非注力": "lightgray",
        },
        symbol_map={
            "3歳": "circle",
            "3歳以上": "diamond",
            "4歳以上": "square",
        },
        hover_name="レース名",
        hover_data={
            "注力レース": True,
            "格": True,
            "年齢": True,
            "トリム後ROI": ":.3f",
            "トリム後的中率": ":.1%",
            "距離": True,
            "件数": True,
            "トリム後件数": True,
            "通常ROI": ":.3f",
            "通常的中率": ":.1%",
            "トリム後収支": True,
            "通常収支": True,
            "1レース予算": True,
            "グラフ色区分": False,
        },
        title="全レース 距離 × ROI（非注力レースはグレー）",
    )

    for trace in fig.data:
        trace_name = str(trace.name)
        if "非注力" in trace_name:
            trace.update(
                marker=dict(
                    size=9,
                    opacity=0.45,
                    color="lightgray",
                    line=dict(width=0.5, color="white"),
                )
            )
        else:
            trace.update(
                marker=dict(
                    size=11,
                    opacity=0.95,
                    line=dict(width=0.8, color="white"),
                )
            )

    fig.update_layout(
        xaxis_title="距離(m)",
        yaxis_title="ROI",
        legend_title_text="格 / 注力区分 / 年齢",
        hovermode="closest",
    )

    fig.write_html(html_path, include_plotlyjs="cdn")
    return fig, plot_df

def print_buy_pattern_master() -> None:
    print("=== 登録している買い方一覧 ===")
    for bet_type in BET_TYPES:
        print(f"\n[{bet_type}]")
        for p in PATTERNS_BY_TYPE[bet_type]:
            print(f" - {p.key}: {p.description}")


def display_priority_races_preview(priority_df: pd.DataFrame, requested_race_names: List[str]) -> None:
    print("\n=== 指定レース 先出し表示 ===")

    if priority_df.empty:
        print("指定したレースは見つかりませんでした。")
        return

    priority_view_df = priority_df[
        [
            "レース名",
            "トリム後件数",
            "1レース予算",
            "トリム後ROI",
            "トリム後的中率",
            "通常ROI",
            "通常的中率",
            "トリム後収支",
            "通常収支",
            "単勝_型",
            "馬連_型",
            "馬単_型",
            "３連複_型",
            "３連単_型",
            "買い方サマリー",
        ]
    ].copy()

    display(
        priority_view_df.style
        .format({
            "トリム後ROI": "{:.3f}",
            "トリム後的中率": "{:.1%}",
            "通常ROI": "{:.3f}",
            "通常的中率": "{:.1%}",
        })
        .hide(axis="index")
    )

    found_races = set(priority_df["レース名"])
    missing_races = [x for x in unique_preserve_order(requested_race_names) if x not in found_races]
    if missing_races:
        print("見つからなかったレース:", ", ".join(missing_races))


def display_all_races_ranking(all_races_ranking_df: pd.DataFrame, upper_trim_count: int) -> None:
    print(f"\n=== 全レース ROI効率ランキング（トリム後ROI順 / 上位{upper_trim_count}件除外） ===")

    if all_races_ranking_df.empty:
        print("対象レースがありませんでした。")
        return

    view_cols = [
        "レース名",
        "注力レース",
        "トリム後件数",
        "1レース予算",
        "トリム後ROI",
        "トリム後的中率",
        "通常ROI",
        "通常的中率",
        "単勝_型",
        "馬連_型",
        "馬単_型",
        "３連複_型",
        "３連単_型",
        "買い方サマリー",
    ]
    view_cols = [c for c in view_cols if c in all_races_ranking_df.columns]
    view_df = all_races_ranking_df[view_cols].copy()

    display(
        view_df.style
        .format({
            "トリム後ROI": "{:.3f}",
            "トリム後的中率": "{:.1%}",
            "通常ROI": "{:.3f}",
            "通常的中率": "{:.1%}",
        })
        .hide(axis="index")
    )

def display_group_roi_using_race_best_patterns(group_df: pd.DataFrame, title: str) -> None:
    print(f"\n=== {title} ===")

    if group_df.empty:
        print("対象データがありませんでした。")
        return

    view_df = group_df[
        [
            "集計値",
            "件数",
            "トリム後件数",
            "使用レース種類数",
            "平均1レース投資",
            "トリム後ROI",
            "トリム後的中率",
            "通常ROI",
            "通常的中率",
            "トリム後収支",
            "通常収支",
            "総投資",
            "総払戻",
            "買い方ルール",
        ]
    ].copy()

    display(
        view_df.style
        .format({
            "トリム後ROI": "{:.3f}",
            "トリム後的中率": "{:.1%}",
            "通常ROI": "{:.3f}",
            "通常的中率": "{:.1%}",
        })
        .hide(axis="index")
    )


def display_all_yearly_profit_summary(
    yearly_summary_df: pd.DataFrame,
    yearly_detail_df: pd.DataFrame,
) -> None:
    print("\n=== 年別: 各レースの最良ROI買い方で買った場合の年間収支 ===")

    if yearly_summary_df.empty:
        print("対象年のデータはありませんでした。")
        return

    display(
        yearly_summary_df[
            ["年", "対象レース種類数", "対象レース件数", "総投資", "総払戻", "総収支", "年間ROI"]
        ].style
        .format({"年間ROI": "{:.3f}"})
        .hide(axis="index")
    )

    # 将来的に使うかもしれないので、年別の全レース明細表示は残してコメントアウト
    # if not yearly_detail_df.empty:
    #     print("\n--- 年別明細 ---")
    #     display(
    #         yearly_detail_df[
    #             ["年", "レース名", "件数", "的中数", "的中率", "総投資", "総払戻", "総収支", "ROI", "買い方サマリー"]
    #         ].style
    #         .format({
    #             "的中率": "{:.1%}",
    #             "ROI": "{:.3f}",
    #         })
    #         .hide(axis="index")
    #     )


def display_monthly_cumulative_trimmed_roi(
    monthly_cumulative_trimmed_roi_df: pd.DataFrame,
    yearly_summary_df: Optional[pd.DataFrame] = None,
) -> None:
    print("\n=== 月別累計 トリム後ROI / 年間収支総計 ===")

    if monthly_cumulative_trimmed_roi_df.empty:
        print("対象データがありませんでした。")
        return

    display_df = build_monthly_cumulative_trimmed_roi_print_table(
        monthly_cumulative_trimmed_roi_df=monthly_cumulative_trimmed_roi_df,
        yearly_summary_df=yearly_summary_df,
    )

    display(
        display_df.style
        .hide(axis="index")
    )


def display_monthly_cumulative_trimmed_roi_range(
    monthly_cumulative_trimmed_roi_range_df: pd.DataFrame,
    yearly_summary_df: Optional[pd.DataFrame] = None,
) -> None:
    print("\n=== 月別累計 トリム後ROI（1月起点）/ 年間収支総計 ===")

    if monthly_cumulative_trimmed_roi_range_df.empty:
        print("対象データがありませんでした。")
        return

    display_df = build_monthly_cumulative_trimmed_roi_print_table(
        monthly_cumulative_trimmed_roi_df=monthly_cumulative_trimmed_roi_range_df,
        yearly_summary_df=yearly_summary_df,
    )

    display(
        display_df.style
        .hide(axis="index")
    )


def display_grade_year_roi_using_race_best_patterns(
    grade_year_roi_df: pd.DataFrame,
    upper_trim_count: int,
) -> None:
    if upper_trim_count > 0:
        print(f"\n=== 格×年別 トリム後ROI（注力レース / 各レース別最適買い方 / 上位{upper_trim_count}件除外 / 総計列あり） ===")
    else:
        print("\n=== 格×年別 ROI（注力レース / 各レース別最適買い方 / 除外なし / 総計列あり） ===")

    if grade_year_roi_df.empty:
        print("対象データがありませんでした。")
        return

    roi_cols = [c for c in grade_year_roi_df.columns if c != "格"]

    display(
        grade_year_roi_df.style
        .format({c: "{:.3f}" for c in roi_cols}, na_rep="-")
        .hide(axis="index")
    )


def display_grade_year_roi_detail_using_race_best_patterns(
    grade_year_roi_detail_df: pd.DataFrame,
) -> None:
    print("\n=== 格×年別 ROI明細（各レース別最適買い方） ===")

    if grade_year_roi_detail_df.empty:
        print("対象データがありませんでした。")
        return

    view_df = grade_year_roi_detail_df[
        [
            "格",
            "年",
            "件数",
            "トリム後件数",
            "使用レース種類数",
            "総投資",
            "総払戻",
            "総収支",
            "ROI",
            "的中率",
            "トリム後ROI",
            "トリム後的中率",
            "買い方ルール",
        ]
    ].copy()

    display(
        view_df.style
        .format({
            "ROI": "{:.3f}",
            "的中率": "{:.1%}",
            "トリム後ROI": "{:.3f}",
            "トリム後的中率": "{:.1%}",
        })
        .hide(axis="index")
    )


# --------------------------------------------
# 実行
# --------------------------------------------
raw_all_df = extract_from_numbers(numbers_path, sheet_keyword=sheet_keyword)
raw_all_df = combine_duplicate_columns(raw_all_df)

all_df = filter_completed_result_rows(raw_all_df)
all_df = combine_duplicate_columns(all_df)

filter_check_df = combine_duplicate_columns(raw_all_df)
empty_first_count = int((filter_check_df["1着"].map(normalize_text) == "").sum())
empty_honmei_count = int((filter_check_df["◎"].map(normalize_text) == "").sum())
excluded_empty_result_count = len(raw_all_df) - len(all_df)

print("\n=== 結果入力フィルタ ===")
print(f"全レコード: {len(raw_all_df):,}件")
print(f"1着・◎あり: {len(all_df):,}件")
print(f"1着空欄: {empty_first_count:,}件")
print(f"◎空欄: {empty_honmei_count:,}件")
print(f"1着または◎空欄で除外: {excluded_empty_result_count:,}件")

json_records = all_df[REQUIRED_COLUMNS + OPTIONAL_COLUMNS].copy()
json_records["年齢"] = json_records["年齢"].astype(str)
json_records = json_records.to_dict(orient="records")

with open(extracted_json_path, "w", encoding="utf-8") as f:
    json.dump(json_records, f, ensure_ascii=False, indent=2)

priority_preview_df = pd.DataFrame()

if PRIORITY_RACE_NAMES:
    priority_preview_df = build_priority_races_preview(
        all_df=all_df,
        priority_race_names=PRIORITY_RACE_NAMES,
        max_budget_per_race=max_budget_per_race,
        step=step,
        upper_trim_count=upper_trim_count,
        roi_target_budget_per_race=ROI_TARGET_BUDGET_PER_RACE,
        show_progress=SHOW_PROGRESS and not PARALLEL,
        progress_interval_sec=PROGRESS_INTERVAL_SEC,
        parallel=PARALLEL,
        max_workers=MAX_WORKERS,
        show_parallel_progress=SHOW_PARALLEL_PROGRESS,
    )
    display_priority_races_preview(priority_preview_df, PRIORITY_RACE_NAMES)

print_buy_pattern_master()

all_races_ranking_df = build_all_races_ranking(
    all_df=all_df,
    max_budget_per_race=max_budget_per_race,
    step=step,
    upper_trim_count=upper_trim_count,
    roi_target_budget_per_race=ROI_TARGET_BUDGET_PER_RACE,
    show_progress=SHOW_PROGRESS and not PARALLEL,
    progress_interval_sec=PROGRESS_INTERVAL_SEC,
    parallel=PARALLEL,
    max_workers=MAX_WORKERS,
    show_parallel_progress=SHOW_PARALLEL_PROGRESS,
)

all_races_ranking_df = add_focus_flag_to_all_races_ranking(
    all_races_ranking_df=all_races_ranking_df,
    min_trimmed_roi=FOCUS_TRIMMED_ROI_MIN,
    min_trimmed_hit_rate=FOCUS_TRIMMED_HIT_RATE_MIN,
)

display_all_races_ranking(all_races_ranking_df, upper_trim_count)

# --------------------------------------------
# 注力レース抽出
# ※ 別表としては表示しない。全レースランキングの「注力レース」列で確認する。
# --------------------------------------------
focus_races_ranking_df = build_focus_race_ranking(
    all_races_ranking_df=all_races_ranking_df,
    min_trimmed_roi=FOCUS_TRIMMED_ROI_MIN,
    min_trimmed_hit_rate=FOCUS_TRIMMED_HIT_RATE_MIN,
)

focus_all_df = filter_all_df_to_ranking_races(
    all_df=all_df,
    ranking_df=focus_races_ranking_df,
)

print("\n=== 注力レース集計対象 ===")
print(f"注力レース種類数: {focus_races_ranking_df['レース名'].nunique() if not focus_races_ranking_df.empty else 0:,}件")
print(f"注力レース対象レコード数: {len(focus_all_df):,}件")

# --------------------------------------------
# 以降の集計は注力レースのみ
# --------------------------------------------

# 格ごとのROIは、格ごとに単一の買い方を最適化しない。
# 各レース名ごとの最適買い方をそのまま使って、格単位に集計する。
# ただし単独表示はせず、格×年別ROI表の「総計」列として統合する。
grade_roi_summary_df = build_group_roi_using_race_best_patterns(
    all_df=focus_all_df,
    race_ranking_df=focus_races_ranking_df,
    group_col="格",
    upper_trim_count=upper_trim_count,
)

# 格×年別ROIも、格ごとの買い方ではなく、各レース名ごとの最適買い方を使用する。
grade_year_roi_df, grade_year_roi_detail_df = build_grade_year_roi_using_race_best_patterns(
    filtered_all_df=focus_all_df,
    race_ranking_df=focus_races_ranking_df,
    target_years=YEARLY_PROFIT_TARGET_YEARS,
    upper_trim_count=RACE_BEST_GRADE_YEAR_UPPER_TRIM_COUNT,
)

grade_year_roi_df = add_grade_total_column_to_grade_year_roi(
    grade_year_roi_df=grade_year_roi_df,
    grade_roi_summary_df=grade_roi_summary_df,
)

display_grade_year_roi_using_race_best_patterns(
    grade_year_roi_df,
    RACE_BEST_GRADE_YEAR_UPPER_TRIM_COUNT,
)

# 格×年別 ROI明細（各レース別最適買い方）は不要なので表示しない
# display_grade_year_roi_detail_using_race_best_patterns(grade_year_roi_detail_df)

# 競馬場ごとのROI集計は削除

# 年別年間収支は単独表示しない。
# 月別累計ROI表の総計行として表示する。
yearly_summary_df, yearly_detail_df = calc_all_yearly_profit_using_best_roi_patterns(
    raw_all_df=raw_all_df,
    filtered_all_df=focus_all_df,
    ranking_df=focus_races_ranking_df,
    target_years=YEARLY_PROFIT_TARGET_YEARS,
)

# display_all_yearly_profit_summary(yearly_summary_df, yearly_detail_df) は不要

monthly_cumulative_trimmed_roi_df, monthly_cumulative_trimmed_roi_detail_df = build_monthly_cumulative_trimmed_roi_table(
    raw_all_df=raw_all_df,
    filtered_all_df=focus_all_df,
    ranking_df=focus_races_ranking_df,
    target_years=MONTHLY_CUMULATIVE_ROI_YEARS,
    target_months=MONTHLY_CUMULATIVE_ROI_MONTHS,
    upper_trim_count=upper_trim_count,
)

display_monthly_cumulative_trimmed_roi(
    monthly_cumulative_trimmed_roi_df=monthly_cumulative_trimmed_roi_df,
    yearly_summary_df=yearly_summary_df,
)

monthly_cumulative_trimmed_roi_range_df = build_monthly_cumulative_trimmed_roi_range_label_table(
    monthly_cumulative_trimmed_roi_df
)

display_monthly_cumulative_trimmed_roi_range(
    monthly_cumulative_trimmed_roi_range_df=monthly_cumulative_trimmed_roi_range_df,
    yearly_summary_df=yearly_summary_df,
)

fig, scatter_source_df = build_scatter_plot(
    ranking_df=all_races_ranking_df,
    all_df=all_df,
    html_path=scatter_html_path,
    source_csv_path=scatter_source_csv_path,
)

output_payload = {
    "search_condition": {
        "max_budget_per_race": max_budget_per_race,
        "step": step,
        "roi_target_budget_per_race": ROI_TARGET_BUDGET_PER_RACE,
        "min_each_by_type": MIN_EACH_BY_TYPE,
        "upper_trim_count": upper_trim_count,
        "race_best_grade_year_upper_trim_count": RACE_BEST_GRADE_YEAR_UPPER_TRIM_COUNT,
        "priority_race_names": PRIORITY_RACE_NAMES,
        "yearly_profit_target_years": YEARLY_PROFIT_TARGET_YEARS,
        "monthly_cumulative_roi_years": (
            get_available_years(raw_all_df)
            if MONTHLY_CUMULATIVE_ROI_YEARS is None
            else MONTHLY_CUMULATIVE_ROI_YEARS
        ),
        "monthly_cumulative_roi_months": MONTHLY_CUMULATIVE_ROI_MONTHS,
        "focus_race_condition": {
            "trimmed_roi_min": FOCUS_TRIMMED_ROI_MIN,
            "trimmed_hit_rate_rule": f">{FOCUS_TRIMMED_HIT_RATE_MIN}",
            "definition": "トリム後ROIが1.5以上、かつトリム後的中率が30%超",
        },
    },
    "prediction_filter": {
        "raw_record_count": len(raw_all_df),
        "used_record_count": len(all_df),
        "empty_first_place_count": empty_first_count,
        "empty_honmei_count": empty_honmei_count,
        "excluded_empty_required_count": excluded_empty_result_count,
        "exclude_rule": "1着列または◎列が空欄の行を除外",
    },
    "optimization_metric": {
        "primary_for_roi_rank": "利益上位レース除外後のROI",
        "primary_for_profit_rank": "利益上位レース除外後の収支",
        "trim_basis": "各レースの最終profit上位を除外。ただし格×年別ROIは RACE_BEST_GRADE_YEAR_UPPER_TRIM_COUNT に従う",
        "roi_budget_scaling": "最適ROI結果の1レース予算が1万円未満なら、1万円を超えない最大の整数倍で増額",
        "group_roi_rule": "格ごとのROIや格×年別ROIは、格単位で買い方を最適化せず、各レース名ごとの最適買い方を使用して集計",
        "focus_race_rule": "全レースROI効率ランキングに注力レース列を付与。トリム後ROI >= 1.5 かつ トリム後的中率 > 0.30 のレースを注力レースとし、以降の集計は注力レースのみで実施",
    },
    "buy_patterns_master": [
        {
            "key": p.key,
            "bet_type": p.bet_type,
            "payout_col": p.payout_col,
            "point_count": p.point_count,
            "description": p.description,
            "hit_rule": p.hit_rule,
            "marks": list(p.marks),
        }
        for p in BUY_PATTERNS
    ],
    "priority_races_preview": (
        priority_preview_df.to_dict(orient="records")
        if not priority_preview_df.empty else []
    ),
    "all_races_roi_ranking": (
        all_races_ranking_df.to_dict(orient="records")
        if not all_races_ranking_df.empty else []
    ),
    "focus_race_record_count": len(focus_all_df),
    "grade_year_roi_by_race_best_patterns": (
        grade_year_roi_df.to_dict(orient="records")
        if not grade_year_roi_df.empty else []
    ),
    "yearly_profit_summary_by_year": (
        yearly_summary_df.to_dict(orient="records")
        if not yearly_summary_df.empty else []
    ),
    "monthly_cumulative_trimmed_roi": (
        monthly_cumulative_trimmed_roi_df.to_dict(orient="records")
        if not monthly_cumulative_trimmed_roi_df.empty else []
    ),
    "monthly_cumulative_trimmed_roi_detail": (
        monthly_cumulative_trimmed_roi_detail_df.to_dict(orient="records")
        if not monthly_cumulative_trimmed_roi_detail_df.empty else []
    ),
    "monthly_cumulative_trimmed_roi_range": (
        monthly_cumulative_trimmed_roi_range_df.to_dict(orient="records")
        if not monthly_cumulative_trimmed_roi_range_df.empty else []
    ),
    "scatter_plot_summary": (
        scatter_source_df[
            ["レース名", "注力レース", "距離", "格", "年齢", "1レース予算", "トリム後ROI", "通常ROI"]
        ].to_dict(orient="records")
        if not scatter_source_df.empty else []
    ),
    "saved_files": {
        "extracted_json": extracted_json_path,
        "ranking_json": ranking_json_path,
        "all_races_csv": all_races_csv_path if not all_races_ranking_df.empty else None,
        "grade_year_roi_by_race_best_patterns_csv": grade_year_roi_csv_path if not grade_year_roi_df.empty else None,
        "monthly_cumulative_trimmed_roi_csv": (
            monthly_cumulative_trimmed_roi_csv_path
            if not monthly_cumulative_trimmed_roi_df.empty else None
        ),
        "monthly_cumulative_trimmed_roi_range_csv": (
            monthly_cumulative_trimmed_roi_range_csv_path
            if not monthly_cumulative_trimmed_roi_range_df.empty else None
        ),
        "scatter_html": scatter_html_path if fig is not None else None,
        "scatter_source_csv": scatter_source_csv_path if not scatter_source_df.empty else None,
    },
}

if not all_races_ranking_df.empty:
    all_races_ranking_df.to_csv(all_races_csv_path, index=False, encoding="utf-8-sig")

# 格ごとROI集計は単独CSV不要
# if not grade_roi_summary_df.empty:
#     grade_roi_summary_df.to_csv(grade_roi_summary_csv_path, index=False, encoding="utf-8-sig")

if not grade_year_roi_df.empty:
    grade_year_roi_df.to_csv(grade_year_roi_csv_path, index=False, encoding="utf-8-sig")

# 格×年別ROI明細は不要
# if not grade_year_roi_detail_df.empty:
#     grade_year_roi_detail_df.to_csv(grade_year_roi_detail_csv_path, index=False, encoding="utf-8-sig")

if not monthly_cumulative_trimmed_roi_df.empty:
    monthly_cumulative_trimmed_roi_df.to_csv(
        monthly_cumulative_trimmed_roi_csv_path,
        index=False,
        encoding="utf-8-sig",
    )

if not monthly_cumulative_trimmed_roi_range_df.empty:
    monthly_cumulative_trimmed_roi_range_df.to_csv(
        monthly_cumulative_trimmed_roi_range_csv_path,
        index=False,
        encoding="utf-8-sig",
    )

with open(ranking_json_path, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2)

if fig is not None:
    print(f"- 散布図HTML: {scatter_html_path}")

if not scatter_source_df.empty:
    print(f"- 散布図元データCSV: {scatter_source_csv_path}")

if fig is not None:
    fig.show()
else:
    print("\n散布図を作れるデータ（距離・格・年齢付きレース）がありませんでした。")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libsnappy-dev:amd64.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../libsnappy-dev_1.1.8-1build3_amd64.deb ...
Unpacking libsnappy-dev:amd64 (1.1.8-1build3) ...
Setting up libsnappy-dev:amd64 (1.1.8-1build3) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 23.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.0 which is incompatib

レース名,トリム後件数,1レース予算,トリム後ROI,トリム後的中率,通常ROI,通常的中率,トリム後収支,通常収支,単勝_型,馬連_型,馬単_型,３連複_型,３連単_型,買い方サマリー
東京優駿,16,8400,3.706,31.2%,6.408,38.9%,363720,817740,,,,３連複_◎抜きBOX4,３連単_◎1着固定4,"３連複: ３連複_◎抜きBOX4 / 1点1,200円 / 4点 / 計4,800円 | ３連単: ３連単_◎1着固定4 / 1点300円 / 12点 / 計3,600円"
目黒記念,16,8700,1.065,31.2%,1.701,38.9%,9030,109800,単勝_◎,馬連_BOX5,馬単_◎1着固定4,３連複_◎抜きBOX4,,"単勝: 単勝_◎ / 1点3,300円 / 1点 / 計3,300円 | 馬連: 馬連_BOX5 / 1点300円 / 10点 / 計3,000円 | 馬単: 馬単_◎1着固定4 / 1点300円 / 4点 / 計1,200円 | ３連複: ３連複_◎抜きBOX4 / 1点300円 / 4点 / 計1,200円"
葵S,2,8400,0.617,50.0%,2.645,75.0%,-6440,55280,単勝_◎,馬連_BOX6,,,,"単勝: 単勝_◎ / 1点2,400円 / 1点 / 計2,400円 | 馬連: 馬連_BOX6 / 1点400円 / 15点 / 計6,000円"


=== 登録している買い方一覧 ===

[単勝]
 - 単勝_◎: ◎ 1点

[馬連]
 - 馬連_◎流し4: ◎-◯▲△☆ 4点流し
 - 馬連_BOX4: ◯▲△☆ 6点BOX
 - 馬連_◎流し5: ◎-◯▲△☆⑥ 5点流し
 - 馬連_BOX5: ◯▲△☆⑥ 10点BOX
 - 馬連_◎流し6: ◎-◯▲△☆⑥⑦ 6点流し
 - 馬連_BOX6: ◯▲△☆⑥⑦ 15点BOX

[馬単]
 - 馬単_◎1着固定4: ◎→◯▲△☆ 4点流し
 - 馬単_◎1着固定5: ◎→◯▲△☆⑥ 5点流し
 - 馬単_◎1着固定6: ◎→◯▲△☆⑥⑦ 6点流し

[３連複]
 - ３連複_◎◯フォーメ4: ◎◯-◎〜☆-◎〜☆ 9点フォーメ
 - ３連複_◎流し4: ◎-◯▲△☆ 6点流し
 - ３連複_◎抜きBOX4: ◯▲△☆ 4点BOX
 - ３連複_◎◯フォーメ5: ◎◯-◎〜⑥-◎〜⑥ 16点フォーメ
 - ３連複_◎流し5: ◎-◯▲△☆⑥ 10点流し
 - ３連複_◎抜きBOX5: ◯▲△☆⑥ 10点BOX
 - ３連複_◎◯フォーメ6: ◎◯-◎〜⑦-◎〜⑦ 25点フォーメ
 - ３連複_◎流し6: ◎-◯▲△☆⑥⑦ 15点流し
 - ３連複_◎抜きBOX6: ◯▲△☆⑥⑦ 20点BOX

[３連単]
 - ３連単_◎1着固定4: ◎→◯▲△☆→◯▲△☆ 12点流し
 - ３連単_◎1着固定5: ◎→◯▲△☆⑥→◯▲△☆⑥ 20点流し
 - ３連単_◎1着固定6: ◎→◯▲△☆⑥⑦→◯▲△☆⑥⑦ 30点流し
並列集計: 152/152 レース完了

=== 全レース ROI効率ランキング（トリム後ROI順 / 上位2件除外） ===


レース名,注力レース,トリム後件数,1レース予算,トリム後ROI,トリム後的中率,通常ROI,通常的中率,単勝_型,馬連_型,馬単_型,３連複_型,３連単_型,買い方サマリー
優駿牝馬,注力,17,9000,7.167,52.9%,20.487,57.9%,,,,３連複_◎流し4,３連単_◎1着固定4,"３連複: ３連複_◎流し4 / 1点900円 / 6点 / 計5,400円 | ３連単: ３連単_◎1着固定4 / 1点300円 / 12点 / 計3,600円"
アルテミスS,注力,10,9000,4.976,40.0%,17.269,50.0%,,,,,３連単_◎1着固定6,"３連単: ３連単_◎1着固定6 / 1点300円 / 30点 / 計9,000円"
エリザベス女王杯,注力,16,9600,4.006,31.2%,12.467,38.9%,,,,,３連単_◎1着固定4,"３連単: ３連単_◎1着固定4 / 1点800円 / 12点 / 計9,600円"
東京優駿,注力,16,8400,3.706,31.2%,6.408,38.9%,,,,３連複_◎抜きBOX4,３連単_◎1着固定4,"３連複: ３連複_◎抜きBOX4 / 1点1,200円 / 4点 / 計4,800円 | ３連単: ３連単_◎1着固定4 / 1点300円 / 12点 / 計3,600円"
小倉記念,注力,16,8100,3.441,37.5%,6.033,44.4%,,馬連_◎流し5,,,３連単_◎1着固定4,"馬連: 馬連_◎流し5 / 1点900円 / 5点 / 計4,500円 | ３連単: ３連単_◎1着固定4 / 1点300円 / 12点 / 計3,600円"
大阪杯,注力,8,9000,3.230,50.0%,9.226,60.0%,,,,,３連単_◎1着固定6,"３連単: ３連単_◎1着固定6 / 1点300円 / 30点 / 計9,000円"
京都記念,注力,17,7800,2.804,70.6%,4.169,73.7%,,,馬単_◎1着固定4,３連複_◎抜きBOX5,３連単_◎1着固定4,"馬単: 馬単_◎1着固定4 / 1点300円 / 4点 / 計1,200円 | ３連複: ３連複_◎抜きBOX5 / 1点300円 / 10点 / 計3,000円 | ３連単: ３連単_◎1着固定4 / 1点300円 / 12点 / 計3,600円"
京王杯スプリングC,注力,17,8800,2.772,35.3%,4.335,42.1%,,,,３連複_◎抜きBOX5,３連単_◎1着固定4,"３連複: ３連複_◎抜きBOX5 / 1点400円 / 10点 / 計4,000円 | ３連単: ３連単_◎1着固定4 / 1点400円 / 12点 / 計4,800円"
京都新聞杯,注力,17,8400,2.699,35.3%,4.595,42.1%,,,馬単_◎1着固定4,,,"馬単: 馬単_◎1着固定4 / 1点2,100円 / 4点 / 計8,400円"
皐月賞,注力,17,9000,2.661,47.1%,3.676,52.6%,,馬連_◎流し4,,３連複_◎流し4,,"馬連: 馬連_◎流し4 / 1点900円 / 4点 / 計3,600円 | ３連複: ３連複_◎流し4 / 1点900円 / 6点 / 計5,400円"



=== 注力レース集計対象 ===
注力レース種類数: 47件
注力レース対象レコード数: 758件

=== 格×年別 ROI（注力レース / 各レース別最適買い方 / 除外なし / 総計列あり） ===


...


=== 月別累計 トリム後ROI / 年間収支総計 ===


月,2008年,2009年,2010年,2011年,2012年,2013年,2014年,2015年,2016年,2017年,2018年,2019年,2020年,2021年,2022年,2023年,2024年,2025年,2026年
1月累計,0.000,0.000,0.000,3.780,0.000,0.000,0.000,0.000,0.000,2.619,1.587,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2月累計,0.000,0.212,0.248,3.250,0.482,2.149,0.437,0.000,0.123,6.004,1.531,1.724,0.000,1.378,0.000,0.471,0.000,3.227,0.000
3月累計,0.203,0.620,0.536,3.599,0.834,2.215,0.676,1.779,0.453,5.006,2.268,1.955,0.303,1.960,0.250,1.172,2.215,3.254,0.131
4月累計,1.792,1.190,2.216,2.679,1.031,3.165,1.655,1.230,0.474,3.996,2.433,2.116,1.437,3.304,0.160,2.976,2.694,2.896,0.292
5月累計,2.664,0.928,3.493,4.495,2.514,3.121,2.244,2.237,1.776,3.888,2.273,2.516,2.621,2.715,1.448,3.033,2.665,1.945,0.365
6月累計,2.746,0.836,3.237,4.180,2.401,3.641,2.605,2.542,1.511,4.092,2.365,2.260,2.397,4.009,1.405,3.279,2.716,1.728,0.365
7月累計,2.746,0.836,3.237,3.996,2.401,3.641,3.106,2.542,1.511,4.092,2.365,2.260,2.397,4.009,1.405,3.279,2.716,1.667,0.365
8月累計,2.547,0.944,3.439,3.896,2.822,3.279,2.976,2.634,1.713,3.636,2.143,2.859,2.224,4.412,1.665,3.276,2.616,1.551,0.365
9月累計,2.521,1.436,3.358,3.725,2.822,3.279,2.976,2.634,1.713,3.636,2.143,2.859,2.224,4.412,1.665,3.276,2.616,1.551,0.365
10月累計,2.292,1.483,3.164,3.370,2.537,3.354,2.754,2.524,1.498,4.501,2.205,2.761,3.036,5.448,2.104,3.196,2.392,1.505,0.365



=== 月別累計 トリム後ROI（1月起点）/ 年間収支総計 ===


月,2008年,2009年,2010年,2011年,2012年,2013年,2014年,2015年,2016年,2017年,2018年,2019年,2020年,2021年,2022年,2023年,2024年,2025年,2026年
1月累計,0.000,0.000,0.000,3.780,0.000,0.000,0.000,0.000,0.000,2.619,1.587,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1-2月累計,0.000,0.212,0.248,3.250,0.482,2.149,0.437,0.000,0.123,6.004,1.531,1.724,0.000,1.378,0.000,0.471,0.000,3.227,0.000
1-3月累計,0.203,0.620,0.536,3.599,0.834,2.215,0.676,1.779,0.453,5.006,2.268,1.955,0.303,1.960,0.250,1.172,2.215,3.254,0.131
1-4月累計,1.792,1.190,2.216,2.679,1.031,3.165,1.655,1.230,0.474,3.996,2.433,2.116,1.437,3.304,0.160,2.976,2.694,2.896,0.292
1-5月累計,2.664,0.928,3.493,4.495,2.514,3.121,2.244,2.237,1.776,3.888,2.273,2.516,2.621,2.715,1.448,3.033,2.665,1.945,0.365
1-6月累計,2.746,0.836,3.237,4.180,2.401,3.641,2.605,2.542,1.511,4.092,2.365,2.260,2.397,4.009,1.405,3.279,2.716,1.728,0.365
1-7月累計,2.746,0.836,3.237,3.996,2.401,3.641,3.106,2.542,1.511,4.092,2.365,2.260,2.397,4.009,1.405,3.279,2.716,1.667,0.365
1-8月累計,2.547,0.944,3.439,3.896,2.822,3.279,2.976,2.634,1.713,3.636,2.143,2.859,2.224,4.412,1.665,3.276,2.616,1.551,0.365
1-9月累計,2.521,1.436,3.358,3.725,2.822,3.279,2.976,2.634,1.713,3.636,2.143,2.859,2.224,4.412,1.665,3.276,2.616,1.551,0.365
1-10月累計,2.292,1.483,3.164,3.370,2.537,3.354,2.754,2.524,1.498,4.501,2.205,2.761,3.036,5.448,2.104,3.196,2.392,1.505,0.365


- 散布図HTML: race_distance_roi_scatter.html
- 散布図元データCSV: race_distance_roi_scatter_source.csv
